In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import cudf


In [3]:
df = cudf.read_csv("https://www.dropbox.com/scl/fi/1ey4norrwpvjvz4oj656g/Books_rating.csv?rlkey=k1eiy9jph0y2iu9kbofastsq7&st=rnyydn6b&dl=1")

In [4]:
df

,Id,Title,Price,User_id,profileName,review/helpfulness,review/score,review/time,review/summary,review/text
0,1882931173,Its Only Art If Its Well Hung!,<NA>,AVCGYZL8FQQTD,"Jim of Oz ""jim-of-oz""",7/7,4.0,940636800,Nice collection of Julie Strain images,This is only for Julie Strain fans. It's a col...
1,0826414346,Dr. Seuss: American Icon,<NA>,A30TK6U7DNS82R,Kevin Killian,10/10,5.0,1095724800,Really Enjoyed It,I don't care much for Dr. Seuss but after read...
2,0826414346,Dr. Seuss: American Icon,<NA>,A3UH4UZ4RSVO82,John Granger,10/11,5.0,1078790400,Essential for every personal and Public Library,"If people become the books they read and if ""t..."
3,0826414346,Dr. Seuss: American Icon,<NA>,A2MVUWT453QH61,"Roy E. Perry ""amateur philosopher""",7/7,4.0,1090713600,Phlip Nel gives silly Seuss a serious treatment,"Theodore Seuss Geisel (1904-1991), aka &quot;D..."
4,0826414346,Dr. Seuss: American Icon,<NA>,A22X4XUPKF66MR,"D. H. Richards ""ninthwavestore""",3/3,4.0,1107993600,Good academic overview,Philip Nel - Dr. Seuss: American IconThis is b...
...,...,...,...,...,...,...,...,...,...,...
2999995,B000NSLVCU,The Idea of History,<NA>,<NA>,<NA>,14/19,4.0,937612800,Difficult,"This is an extremely difficult book to digest,..."
2999996,B000NSLVCU,The Idea of History,<NA>,A1SMUB9ASL5L9Y,jafrank,1/1,4.0,1331683200,Quite good and ahead of its time occasionally,This is pretty interesting. Collingwood seems ...
2999997,B000NSLVCU,The Idea of History,<NA>,A2AQMEKZKK5EE4,"L. L. Poulos ""Muslim Mom""",0/0,4.0,1180224000,Easier reads of those not well versed in histo...,"This is a good book but very esoteric. ""What i..."
2999998,B000NSLVCU,The Idea of History,<NA>,A18SQGYBKS852K,"Julia A. Klein ""knitting rat""",1/11,5.0,1163030400,"Yes, it is cheaper than the University Bookstore","My daughter, a freshman at Indiana University,..."


In [5]:
df.describe()


,Price,review/score,review/time
count,481171.000000,3.000000e+06,3.000000e+06
mean,21.762656,4.215289e+00,1.132307e+09
std,26.206541,1.203054e+00,1.493202e+08
min,1.000000,1.000000e+00,-1.000000e+00
25%,10.780000,4.000000e+00,9.999072e+08
50%,14.930000,5.000000e+00,1.128298e+09
75%,23.950000,5.000000e+00,1.269130e+09
max,995.000000,5.000000e+00,1.362355e+09


In [6]:
df[df["review/score"]==4.0]

,Id,Title,Price,User_id,profileName,review/helpfulness,review/score,review/time,review/summary,review/text
0,1882931173,Its Only Art If Its Well Hung!,<NA>,AVCGYZL8FQQTD,"Jim of Oz ""jim-of-oz""",7/7,4.0,940636800,Nice collection of Julie Strain images,This is only for Julie Strain fans. It's a col...
3,0826414346,Dr. Seuss: American Icon,<NA>,A2MVUWT453QH61,"Roy E. Perry ""amateur philosopher""",7/7,4.0,1090713600,Phlip Nel gives silly Seuss a serious treatment,"Theodore Seuss Geisel (1904-1991), aka &quot;D..."
4,0826414346,Dr. Seuss: American Icon,<NA>,A22X4XUPKF66MR,"D. H. Richards ""ninthwavestore""",3/3,4.0,1107993600,Good academic overview,Philip Nel - Dr. Seuss: American IconThis is b...
5,0826414346,Dr. Seuss: American Icon,<NA>,A2F6NONFUDB6UK,Malvin,2/2,4.0,1127174400,One of America's greatest creative talents,"""Dr. Seuss: American Icon"" by Philip Nel is a ..."
9,0826414346,Dr. Seuss: American Icon,<NA>,A3VA4XFS5WNJO3,Donald Burnside,3/5,4.0,1076371200,Fascinating account of a genius at work,"As far as I am aware, this is the first book-l..."
...,...,...,...,...,...,...,...,...,...,...
2999988,0255364520,An End to Welfare Rights: The Rediscovery of I...,18.95,A25JH6CO4DVINS,Junglies,0/0,4.0,1045526400,Heaven helps those who help themselves.,Another book on welfare reform. Dr. Green invo...
2999994,B000NSLVCU,The Idea of History,<NA>,AOFGOUMXLMVZS,"S. Grotzke ""scquest""",3/3,4.0,1342483200,Thoughtful Critic of History,History is not a scientific process of cutting...
2999995,B000NSLVCU,The Idea of History,<NA>,<NA>,<NA>,14/19,4.0,937612800,Difficult,"This is an extremely difficult book to digest,..."
2999996,B000NSLVCU,The Idea of History,<NA>,A1SMUB9ASL5L9Y,jafrank,1/1,4.0,1331683200,Quite good and ahead of its time occasionally,This is pretty interesting. Collingwood seems ...


In [7]:
import numba.cuda
print("¿GPU disponible?", numba.cuda.is_available())

¿GPU disponible? True


In [19]:
import cudf
import time

df = cudf.DataFrame({
    'a': range(10_000_000),
    'b': range(10_000_000)
})

start = time.time()
df['c'] = df['a'] * df['b']
end = time.time()

print("Tiempo:", end - start)


Tiempo: 0.03480267524719238


In [20]:
df = cudf.DataFrame({
    'a': range(100_000_000),
    'b': range(100_000_000)
})

In [21]:
# 1. Verificar disponibilidad de GPU
import numba.cuda
print("GPU disponible:", numba.cuda.is_available())

GPU disponible: True


In [2]:
# 2. Operación con cuDF
import cudf
import time

print("\n---- cuDF (GPU) ----")
df_cu = cudf.DataFrame({
    'a': range(100_000_000),
    'b': range(100_000_000)
})

start = time.time()
df_cu['c'] = df_cu['a'] * df_cu['b']
end = time.time()
print("Tiempo cuDF:", end - start)



---- cuDF (GPU) ----
Tiempo cuDF: 0.055464982986450195


In [2]:
# 3. Operación equivalente con pandas (CPU)
import pandas as pd

print("\n---- pandas (CPU) ----")
df_pd = pd.DataFrame({
    'a': range(100_000_000),
    'b': range(100_000_000)
})

start = time.time()
df_pd['c'] = df_pd['a'] * df_pd['b']
end = time.time()
print("Tiempo pandas:", end - start)


---- pandas (CPU) ----
Tiempo pandas: 0.4047417640686035


In [5]:
!nvidia-smi


Thu May  1 17:51:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             26W /   70W |    2399MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!pip install cupy-cuda12x

In [11]:
import cupy as cp

df = cudf.DataFrame({'a': cp.random.randint(0, 100000, 50_000_000)})
df = df.sort_values(by='a')
df

,a
9793,0
80069,0
411681,0
610180,0
690756,0
...,...
49690158,99999
49718406,99999
49739245,99999
49763857,99999
